# <font color="#418FDE" size="6.5" uppercase>**Tree Modeling**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Fit decision tree classifiers and regressors for nonlinear prediction tasks. 
- Configure split criteria, depth, leaf-size, and weighting controls. 
- Visualize and interpret fitted tree rules and predictions. 


## **1. Tree Estimators**

### **1.1. Classification Trees**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_A/image_01_01.jpg?v=1783970408" width="250">



>* Predict categories using feature-based decision rules
>* Capture nonlinear patterns across different data regions

>* Splits create purer class groups
>* Recursive rules reveal feature interactions

>* Leaves predict classes and estimated probabilities
>* Limit depth to avoid overfitting



In [ ]:
#@title Python Code - Classification Trees

# Build a tiny classification tree example.
# Use simple rules for beginner learning.
# Visualize nonlinear class regions clearly.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Create a small nonlinear classification dataset.
rng = np.random.default_rng(seed=7)
height_cm = rng.normal(loc=170, scale=12, size=80)
activity_miles = rng.normal(loc=3.0, scale=1.2, size=80)

# Combine features into one training matrix.
X = np.column_stack((height_cm, activity_miles))
y = ((height_cm > 172) & (activity_miles > 3.1)).astype(int)

# Add a second nonlinear positive region.
y = np.where((height_cm < 162) & (activity_miles > 4.0), 1, y)
assert X.shape == (80, 2) and y.shape == (80,)

# Fit a shallow classification tree.
model = DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, random_state=7)
model.fit(X, y)
accuracy = model.score(X, y)

# Predict one new person's class probability.
new_person = np.array([[168, 4.2]])
probability = model.predict_proba(new_person)[0, 1]
prediction = model.predict(new_person)[0]

# Print only a few teaching results.
print(f"scikit-learn version: {sklearn_version}")
print(f"Training accuracy: {accuracy:.2f}")
print(f"New person predicted class: {prediction}")
print(f"Estimated positive probability: {probability:.2f}")

# Plot the fitted tree rules.
plt.figure(figsize=(10, 5))
plot_tree(model, feature_names=["height_cm", "activity_miles"], class_names=["No", "Yes"], filled=True, rounded=True)
plt.title("Classification tree rules")
plt.show()



### **1.2. Regression Trees**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_A/image_01_02.jpg?v=1783970410" width="250">



>* Predict numeric outcomes using feature-based splits
>* Leaves provide predictions for nonlinear patterns

>* Captures interactions and threshold effects
>* Learns local rules for different situations

>* Limit tree depth to prevent overfitting
>* Interpret predictions through learned decision rules



In [ ]:
#@title Python Code - Regression Trees

# Regression trees predict numeric outcomes.
# This example builds one manually.
# Small data keeps behavior understandable.

import numpy as np
import matplotlib.pyplot as plt

# Create deterministic nonlinear training data.
rng = np.random.default_rng(7)

# Use square feet as one feature.
sqft = np.linspace(600, 3000, 36)

# Build a stepped nonlinear price pattern.
base = np.where(sqft < 1400, 180, 260)

# Add another threshold for larger homes.
base = np.where(sqft > 2300, 360, base)

# Add small random noise.
noise = rng.normal(0, 18, sqft.size)

# Store target prices in thousands.
price = base + 0.035 * sqft + noise

# Validate matching one dimensional arrays.
assert sqft.ndim == price.ndim == 1
assert sqft.size == price.size

# Define squared error for one leaf.
def leaf_error(values):
    mean_value = values.mean()
    return np.sum((values - mean_value) ** 2)

# Find the best single split.
def best_split(x_values, y_values):
    candidates = (x_values[:-1] + x_values[1:]) / 2
    best_cut, best_error = None, float("inf")

    for cut in candidates:
        left = y_values[x_values <= cut]
        right = y_values[x_values > cut]

        if left.size < 4 or right.size < 4:
            continue
        error = leaf_error(left) + leaf_error(right)

        if error < best_error:
            best_cut, best_error = cut, error
    return best_cut, best_error

# Fit a tiny depth two regression tree.
root_cut, root_error = best_split(sqft, price)
left_mask = sqft <= root_cut
right_mask = sqft > root_cut

# Split each branch once more.
left_cut, left_error = best_split(sqft[left_mask], price[left_mask])
right_cut, right_error = best_split(sqft[right_mask], price[right_mask])

# Predict using learned leaf averages.
def predict_one(area):
    if area <= root_cut:
        branch_mask = left_mask
        cut_value = left_cut
        side_mask = sqft <= cut_value
    else:
        branch_mask = right_mask
        cut_value = right_cut
        side_mask = sqft <= cut_value

    if area <= cut_value:
        final_mask = branch_mask & side_mask
    else:
        final_mask = branch_mask & ~side_mask
    return price[final_mask].mean()

# Create smooth inputs for visualization.
grid = np.linspace(sqft.min(), sqft.max(), 200)
predictions = np.array([predict_one(value) for value in grid])

# Print a compact model summary.
print(f"Root split: {root_cut:.0f} square feet")
print(f"Left split: {left_cut:.0f} square feet")
print(f"Right split: {right_cut:.0f} square feet")
print(f"Prediction at 1800 sqft: ${predict_one(1800):.0f}k")

# Plot observations and stepwise predictions.
plt.figure(figsize=(7, 4))
plt.scatter(sqft, price, label="training homes")
plt.plot(grid, predictions, color="red", label="tree prediction")
plt.xlabel("Home size, square feet")
plt.ylabel("Price, thousands of dollars")
plt.title("Manual Regression Tree With Threshold Rules")
plt.legend()
plt.tight_layout()
plt.show()



### **1.3. Multioutput Tree Models**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_A/image_01_03.jpg?v=1783970412" width="250">



>* Predict multiple related targets together
>* Leaves store vectors of predictions

>* Shared splits capture related outcome patterns
>* One model gives distinct target predictions

>* Shared rules guide all target predictions
>* Leaves predict separately, with possible compromises



## **2. Split Controls**

### **2.1. Classification Split Criteria**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_A/image_02_01.jpg?v=1783970398" width="250">



>* Choose splits that group similar classes
>* Criteria shape tree structure and patterns

>* Gini and entropy measure node label mixing
>* Both guide cleaner, more informative splits

>* Match criteria to class balance and costs
>* Validate useful, interpretable, decision-aligned branches



In [ ]:
#@title Python Code - Classification Split Criteria

# Compare split criteria for classification trees.
# Tiny data keeps the example transparent.
# Manual calculations avoid external machine learning libraries.

import numpy as np
import matplotlib.pyplot as plt

# Create one feature and binary labels.
x = np.array([1, 2, 3, 4, 5, 6, 7, 8])
y = np.array([0, 0, 0, 1, 0, 1, 1, 1])

# Validate matching one dimensional arrays.
assert x.ndim == y.ndim == 1
assert len(x) == len(y) and len(x) > 2

# Define Gini impurity for class labels.
def gini(labels):
    values, counts = np.unique(labels, return_counts=True)
    probs = counts / counts.sum()

    return 1.0 - np.sum(probs ** 2)

# Define entropy impurity for class labels.
def entropy(labels):
    values, counts = np.unique(labels, return_counts=True)
    probs = counts / counts.sum()

    return -np.sum(probs * np.log2(probs))

# Score every possible threshold split.
def weighted_impurity(threshold, criterion):
    left = y[x <= threshold]
    right = y[x > threshold]

    left_weight = len(left) / len(y)
    right_weight = len(right) / len(y)

    return left_weight * criterion(left) + right_weight * criterion(right)

# Candidate thresholds sit between sorted values.
thresholds = (x[:-1] + x[1:]) / 2
criteria = {"gini": gini, "entropy": entropy}

# Compute best threshold for each criterion.
results = {}
for name, function in criteria.items():
    scores = [weighted_impurity(t, function) for t in thresholds]

    best_index = int(np.argmin(scores))
    results[name] = (thresholds[best_index], scores[best_index], scores)

# Print a compact comparison table.
print("Criterion | best threshold | weighted impurity")
for name, values in results.items():
    print(f"{name:8s} | {values[0]:14.1f} | {values[1]:17.3f}")

# Plot impurity curves for both criteria.
plt.figure(figsize=(7, 4))
for name, values in results.items():
    plt.plot(thresholds, values[2], marker="o", label=name)

# Label the teaching plot clearly.
plt.xlabel("Candidate threshold on feature x")
plt.ylabel("Weighted child-node impurity")
plt.title("Classification split criteria compare candidate splits")
plt.legend()

# Display the single required plot.
plt.tight_layout()
plt.show()



### **2.2. Regression Split Criteria**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_A/image_02_02.jpg?v=1783970395" width="250">



>* Split numeric targets into similar-value groups
>* Leaves predict averages with reduced variation

>* Squared error emphasizes large prediction mistakes
>* Absolute error better handles outliers

>* Tune criteria with depth, leaves, and weights
>* Balance fit, stability, and prediction costs



In [ ]:
#@title Python Code - Regression Split Criteria

# Compare regression tree split criteria.
# Build tiny trees from scratch.
# Visualize predictions and outlier effects.

import numpy as np
import matplotlib.pyplot as plt

# Create a small nonlinear regression dataset.
rng = np.random.default_rng(7)
x = np.linspace(0, 10, 28)
y = np.sin(x) + 0.15 * rng.normal(size=x.size)

# Add one extreme target outlier.
y[-2] = 4.0
X = x.reshape(-1, 1)
assert X.shape[0] == y.size

# Define squared and absolute split losses.
def split_loss(values, criterion):
    if criterion == "squared_error":
        center = np.mean(values)
        return np.sum((values - center) ** 2)
    center = np.median(values)
    return np.sum(np.abs(values - center))

# Find the best one-level split.
def best_stump(x_values, targets, criterion):
    order = np.argsort(x_values)
    xs, ys = x_values[order], targets[order]
    best = (np.inf, None, None, None)

    for i in range(3, len(xs) - 3):
        threshold = (xs[i - 1] + xs[i]) / 2
        left, right = ys[:i], ys[i:]
        loss = split_loss(left, criterion) + split_loss(right, criterion)
        if loss < best[0]:
            best = (loss, threshold, left, right)

    return best

# Compute predictions for each criterion.
criteria = ["squared_error", "absolute_error"]
colors = ["tab:blue", "tab:orange"]
results = []

for criterion in criteria:
    loss, threshold, left, right = best_stump(x, y, criterion)
    left_pred = np.mean(left) if criterion == "squared_error" else np.median(left)
    right_pred = np.mean(right) if criterion == "squared_error" else np.median(right)
    results.append((criterion, threshold, left_pred, right_pred, loss))

# Print a compact comparison table.
print("Criterion comparison for one regression split")
for criterion, threshold, left_pred, right_pred, loss in results:
    print(f"{criterion}: split x <= {threshold:.2f}, left={left_pred:.2f}, right={right_pred:.2f}")

# Plot data and fitted stump predictions.
plt.figure(figsize=(8, 4.5))
plt.scatter(x, y, color="black", label="training points")

# Draw each criterion's piecewise prediction.
for (criterion, threshold, left_pred, right_pred, loss), color in zip(results, colors):
    yhat = np.where(x <= threshold, left_pred, right_pred)
    plt.step(x, yhat, where="mid", color=color, label=criterion)

# Label the plot for interpretation.
plt.axvline(results[0][1], color="tab:blue", linestyle="--", alpha=0.5)
plt.axvline(results[1][1], color="tab:orange", linestyle="--", alpha=0.5)
plt.title("Regression split criteria react differently to an outlier")
plt.xlabel("feature value")
plt.ylabel("numeric target")
plt.legend()
plt.tight_layout()
plt.show()



### **2.3. Depth and Leaf Controls**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_A/image_02_03.jpg?v=1783970400" width="250">



>* Limit tree depth to reduce overfitting
>* Balance flexible patterns with simple rules

>* Require enough data for each leaf
>* Balance stability against finer local patterns

>* Tune depth and leaf sizes together
>* Balance accuracy, stability, and interpretability



## **3. Tree Interpretation**

### **3.1. Class Weight Effects**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_A/image_03_01.jpg?v=1783970402" width="250">



>* Weights make some classes influence splits more
>* They reshape rules for imbalanced outcomes

>* Weights can override raw class counts
>* Interpret rules as cost-sensitive priorities

>* Weights shape sensitivity and false alarm tradeoffs
>* Inspect rules for goals and generalization



### **3.2. Handling Missing Values**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_A/image_03_02.jpg?v=1783970404" width="250">



>* Missing values complicate tree split interpretation
>* Handling choices can hide model assumptions

>* Imputation shapes tree training and visualization
>* Interpret imputed splits as data-handling choices

>* Trees can route missing values during splits
>* Interpret paths as measured and missingness decisions



### **3.3. Visualizing Tree Structure**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_A/image_03_03.jpg?v=1783970406" width="250">



>* Tree diagrams map feature questions to predictions
>* Root-to-leaf paths explain data partitions

>* Check leaf size and class purity
>* Beware deep paths that capture noise

>* Explain tree decisions to nontechnical audiences
>* Check rules for fairness and limits



# <font color="#418FDE" size="6.5" uppercase>**Tree Modeling**</font>


In this lecture, you learned to:
- Fit decision tree classifiers and regressors for nonlinear prediction tasks. 
- Configure split criteria, depth, leaf-size, and weighting controls. 
- Visualize and interpret fitted tree rules and predictions. 

In the next Lecture (Lecture B), we will go over 'Tree Regularization'